In [ ]:
# input
pred_file = "./tmp/idp.pred"
low_pred_file = "./tmp/low_plddt_high_proba.tsv"
low_pred_fasta = "./tmp/low_plddt_high_proba.fasta"
metaldb_pred_site = "../../data/pred_site_plddt_proba.tsv"
# output
idp_mbp_file = "./tmp/150_idp_mbp.tsv"
idp_mbp_fasta = "./tmp/150_idp_mbp.fasta"

In [2]:
import pandas as pd

df = pd.read_table(metaldb_pred_site)
df = df.drop_duplicates(subset=['seq_id'], keep=False)
single_site_ids = set(df['seq_id'])
del df

In [3]:
records = []
with open(pred_file, "r") as f:
    lines = f.readlines()[15:]
    cur_id = None
    for l in lines:
        line = l.strip()
        if len(line) == 0: 
            cur_id = None
            continue

        if l.startswith("#>AFDB"):
            cur_id = l.split()[0].split("-")[1]
        else:
            assert cur_id is not None
            info = line.split()
            seq_num = int(info[0])
            resi = info[1]
            proba = float(info[2])
            records.append({
                "seq_id": cur_id,
                "seq_num": seq_num,
                "resi": resi,
                "proba": proba
            })

In [4]:
df = pd.DataFrame(records)
df = df[df['proba'] > 0.5]
df['posi'] = df['seq_num'].map(lambda x: x - 1)
idrs = set(zip(df['seq_id'], df['posi']))

In [ ]:
df = pd.read_table(low_pred_file)
passed_ids = set()
for _, row in df.iterrows():
    posis = [int(i) - 1 for i in row['site'].split(",")]
    if all([(row['seq_id'], p) in idrs for p in posis]):
        passed_ids.add(row['seq_id'])
df = df[df['seq_id'].map(lambda x: x in passed_ids and x in single_site_ids)]
len(df)

3962

In [6]:
df = df.sample(n=150, random_state=42)
df.to_csv(idp_mbp_file, sep="\t", index=None)

In [7]:
import os
cmd = f"""awk 'NR>1 {{print "AFDB:AF-"$1"-F1"}}' {idp_mbp_file} | seqkit grep -f - {low_pred_fasta} > {idp_mbp_fasta}"""
os.system(cmd)

[INFO] 150 patterns loaded from file


0